# Splitmaa FunctionGemma Manual v4 Training

This notebook is a visual runner for the final `manual_v4` dataset workflow:

- inspect the frozen dataset report
- run strict validation and semantic audit
- convert/train FunctionGemma LoRA
- capture predictions from the trained adapter
- score the locked test set
- inspect failures and metrics

Run it from the repository root: `C:\\Users\\mario\\Documents\\projects\\Splitmaa`.

## 1. Setup

The notebook uses the existing `.venv-train` environment and repo scripts. If a cell fails because a package is missing, install it into `.venv-train`, not a random global Python.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys
from datetime import datetime

REPO = Path.cwd()
DATASET_DIR = REPO / "datasets" / "splitmaa_functiongemma" / "manual_v4"
TRAIN_JSONL = DATASET_DIR / "train.jsonl"
VALIDATION_JSONL = DATASET_DIR / "validation.jsonl"
TEST_JSONL = DATASET_DIR / "test.jsonl"
TRAIN_FG = DATASET_DIR / "train.functiongemma.jsonl"
VALIDATION_FG = DATASET_DIR / "validation.functiongemma.jsonl"
REPORTS_DIR = REPO / "reports" / "functiongemma_eval"
OUTPUT_DIR = REPO / "outputs" / "functiongemma-splitmaa-manual-v4-lora"

PY = REPO / ".venv-train" / "Scripts" / "python.exe"
if not PY.exists():
    PY = Path(sys.executable)

print("repo:", REPO)
print("python:", PY)
print("dataset:", DATASET_DIR)
print("time:", datetime.now().isoformat(timespec="seconds"))

In [ ]:
def run(cmd, *, check=True, timeout=None):
    """Run a command from the repo root and print stdout/stderr."""
    print("\n$", " ".join(str(part) for part in cmd))
    result = subprocess.run(
        [str(part) for part in cmd],
        cwd=REPO,
        text=True,
        capture_output=True,
        timeout=timeout,
    )
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if check and result.returncode != 0:
        raise RuntimeError(f"command failed with exit code {result.returncode}")
    return result

## 2. Dataset Overview

In [ ]:
from collections import Counter

def load_jsonl(path):
    return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]

splits = {
    "train": load_jsonl(TRAIN_JSONL),
    "validation": load_jsonl(VALIDATION_JSONL),
    "test": load_jsonl(TEST_JSONL),
}

for split, rows in splits.items():
    counts = Counter(row["expected"]["arguments"]["workflowType"] for row in rows)
    print(split, len(rows), dict(counts))

print("total", sum(len(rows) for rows in splits.values()))

In [ ]:
try:
    import pandas as pd
    import matplotlib.pyplot as plt
except Exception as exc:
    print("Install optional notebook packages if you want charts:", exc)
else:
    rows = []
    for split, items in splits.items():
        for item in items:
            rows.append({
                "split": split,
                "workflowType": item["expected"]["arguments"]["workflowType"],
                "inputWords": len(item["input"].split()),
                "operations": len(item["expected"]["arguments"].get("operations", [])),
            })
    df = pd.DataFrame(rows)
    display(df.groupby(["split", "workflowType"]).size().unstack(fill_value=0))
    ax = df["workflowType"].value_counts().plot(kind="bar", title="Workflow Distribution", figsize=(10, 4))
    ax.set_xlabel("workflowType")
    ax.set_ylabel("rows")
    plt.tight_layout()
    plt.show()

In [ ]:
report_path = DATASET_DIR / "dataset_report.json"
if report_path.exists():
    report = json.loads(report_path.read_text(encoding="utf-8"))
    print(json.dumps({
        "rows": report.get("rows"),
        "splitCounts": report.get("splitCounts"),
        "workflowCounts": report.get("workflowCounts"),
        "avgInputWords": report.get("avgInputWords"),
        "maxInputWords": report.get("maxInputWords"),
        "maxOperations": report.get("maxOperations"),
    }, indent=2))
else:
    print("No dataset report yet. Run the report cell below.")

## 3. Required Dataset Gates

These should pass before training. They are fast.

In [ ]:
run([
    PY,
    "tools/finetune/validate_splitmaa_dataset.py",
    "--strict-routing",
    TRAIN_JSONL,
    VALIDATION_JSONL,
    TEST_JSONL,
])

In [ ]:
run([
    PY,
    "tools/finetune/semantic_audit_dataset.py",
    TRAIN_JSONL,
    VALIDATION_JSONL,
    TEST_JSONL,
    "--report",
    DATASET_DIR / "semantic_audit_manual_v4.json",
    "--fail-on-blocking",
])

In [ ]:
run([
    PY,
    "tools/finetune/report_splitmaa_dataset.py",
    "--base",
    DATASET_DIR,
])

In [ ]:
run([
    PY,
    "tools/evals/run_eval.py",
    "--dataset",
    TEST_JSONL,
    "--self-test-with-expected",
    "--report",
    DATASET_DIR / "eval_self_test_report.json",
    "--failure-limit",
    "200",
])

## 4. Convert To FunctionGemma Format

Run this whenever the staging JSONL changes. The final files are what training uses.

In [ ]:
run([PY, "tools/finetune/convert_to_functiongemma.py", TRAIN_JSONL, TRAIN_FG])
run([PY, "tools/finetune/convert_to_functiongemma.py", VALIDATION_JSONL, VALIDATION_FG])

## 5. Train LoRA

Start with 3 epochs. If eval shows underfitting, try 5 epochs. Avoid jumping to 8 epochs until we inspect predictions.

In [ ]:
TRAIN_ARGS = [
    PY,
    "tools/finetune/train_functiongemma_sft.py",
    "--base-model", "google/functiongemma-270m-it",
    "--train", TRAIN_FG,
    "--validation", VALIDATION_FG,
    "--output-dir", OUTPUT_DIR,
    "--trainer-backend", "lean",
    "--training-mode", "lora",
    "--lora-r", "8",
    "--lora-alpha", "16",
    "--lora-dropout", "0.05",
    "--epochs", "3",
    "--batch-size", "1",
    "--eval-batch-size", "1",
    "--gradient-accumulation-steps", "8",
    "--max-length", "1024",
    "--dtype", "bfloat16",
]
print(" ".join(str(x) for x in TRAIN_ARGS))

In [ ]:
# This can take a while. Leave the notebook/kernel running.
run(TRAIN_ARGS)

## 6. Capture Predictions From Trained Adapter

In [ ]:
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
PREDICTIONS = REPORTS_DIR / "manual_v4_lora_predictions.jsonl"

run([
    PY,
    "tools/evals/hf_peft_predictions.py",
    "--dataset",
    TEST_JSONL,
    "--adapter",
    OUTPUT_DIR,
    "--output",
    PREDICTIONS,
    "--device",
    "cuda",
    "--dtype",
    "bfloat16",
])

## 7. Score Locked Test Set

Initial acceptance thresholds:

- schema valid rate >= 0.95
- workflow accuracy >= 0.85
- operation sequence accuracy >= 0.80

In [ ]:
EVAL_REPORT = REPORTS_DIR / "manual_v4_lora_eval.json"

run([
    PY,
    "tools/evals/run_eval.py",
    "--dataset",
    TEST_JSONL,
    "--predictions",
    PREDICTIONS,
    "--report",
    EVAL_REPORT,
    "--fail-under-schema-valid",
    "0.95",
    "--fail-under-workflow",
    "0.85",
    "--fail-under-operation-sequence",
    "0.80",
    "--failure-limit",
    "100",
])

In [ ]:
if EVAL_REPORT.exists():
    eval_report = json.loads(EVAL_REPORT.read_text(encoding="utf-8"))
    keys = [
        "examples",
        "parseableRate",
        "schemaValidRate",
        "workflowAccuracy",
        "operationSequenceAccuracy",
        "exactIntentAccuracy",
        "leafArgumentAccuracy",
        "failures",
    ]
    print(json.dumps({key: eval_report.get(key) for key in keys}, indent=2))
else:
    print("No eval report yet.")

## 8. Inspect Failures

If the eval fails thresholds, inspect examples instead of changing hyperparameters blindly.

In [ ]:
if EVAL_REPORT.exists():
    eval_report = json.loads(EVAL_REPORT.read_text(encoding="utf-8"))
    failures = eval_report.get("failureExamples") or eval_report.get("failuresDetail") or []
    print("failure rows:", len(failures))
    for item in failures[:10]:
        print("\n---")
        print(json.dumps(item, indent=2)[:3000])
else:
    print("Run scoring first.")

## 9. Decision Guide

- If schema validity is low: inspect raw predictions and parser behavior first.
- If workflow accuracy is low: review routing failures and add/repair rows only if the label is wrong or under-covered.
- If operation sequence is low: check multi-step rows and long commands.
- If everything passes: export/merge the adapter and move toward a mobile-loadable FunctionGemma artifact.